# npm Hallucination Detector — Ensemble 모델 학습 (Colab)

**CodeBERT(768차원) + 메타데이터(9차원) → XGBoost 앙상블**

런타임 메뉴 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 저장소 클론 & 패키지 설치

In [7]:
# 저장소 클론
!git clone https://github.com/haneul-dev/npm-hallucination-detector.git
%cd npm-hallucination-detector

Cloning into 'npm-hallucination-detector'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 123 (delta 47), reused 116 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 462.89 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/npm-hallucination-detector/npm-hallucination-detector


In [8]:
# 의존성 설치 (torch는 Colab 기본 설치됨)
!pip install -q transformers xgboost shap loguru imbalanced-learn

In [9]:
import sys
sys.path.insert(0, '.')

import torch
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('디바이스:', 'cuda' if torch.cuda.is_available() else 'cpu')

PyTorch: 2.11.0+cu128
CUDA 사용 가능: True
디바이스: cuda


## 2. 데이터 로드 & 전처리

In [10]:
import os, re, random
import numpy as np
import pandas as pd
from loguru import logger

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_FILES = {
    'maloss':       'data/processed/maloss_npm_malicious.csv',
    'backstabbers': 'data/processed/backstabbers_npm.csv',
    'advisory':     'data/processed/npm_advisory.csv',
    'hallucination':'data/processed/llm_hallucinated_packages.csv',
    'benign':       'data/processed/benign_packages.csv',
}

frames = []
for key, path in DATA_FILES.items():
    if not os.path.exists(path):
        print(f'[SKIP] {path}')
        continue
    df = pd.read_csv(path)
    df['label'] = 0 if key == 'benign' else 1
    frames.append(df)
    print(f'[OK] {key}: {len(df)}건')

full = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['name'])

# 클래스 균형 (각 최대 4,000건)
MAX = 4000
mal = full[full['label']==1].sample(min(MAX, (full['label']==1).sum()), random_state=RANDOM_SEED)
ben = full[full['label']==0].sample(min(MAX, (full['label']==0).sum()), random_state=RANDOM_SEED)
df_balanced = pd.concat([mal, ben]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'\n최종 데이터셋: 악성 {len(mal)}건 / 정상 {len(ben)}건 / 합계 {len(df_balanced)}건')

[OK] maloss: 455건
[OK] backstabbers: 5831건
[OK] advisory: 174건
[OK] hallucination: 700건
[OK] benign: 148건

최종 데이터셋: 악성 4000건 / 정상 102건 / 합계 4102건


## 3. 메타데이터 피처 추출 (9차원)

In [11]:
from difflib import SequenceMatcher

POPULAR = ['react','express','lodash','axios','webpack','babel','eslint',
           'typescript','vue','angular','jquery','moment','chalk','commander',
           'dotenv','fastify','koa','next','vite','prisma']

def name_sim(name):
    return max(SequenceMatcher(None, str(name).lower(), p).ratio() for p in POPULAR)

def suspicious(name):
    patterns = [r'\d{3,}$', r'_{2,}', r'-{2,}', r'^[a-z]{1,2}$']
    return int(any(re.search(p, str(name)) for p in patterns))

def extract_meta(row):
    d = float(row.get('downloads', 0) or 0)
    return [
        np.log1p(d),
        name_sim(row.get('name', '')),
        suspicious(row.get('name', '')),
        int(not row.get('author', '')),
        int(row.get('has_install_script', 0)),
        int(float(row.get('dependencies_count', 0)) > 20),
        min(float(row.get('maintainers_count', 0)), 20),
        min(float(row.get('dependencies_count', 0)), 50),
        int(not bool(row.get('exists', True))),
    ]

meta_X = np.array([extract_meta(row) for _, row in df_balanced.iterrows()], dtype=np.float32)
print(f'메타데이터 피처: {meta_X.shape}')

ValueError: cannot convert float NaN to integer

## 4. CodeBERT 임베딩 (768차원)

GPU 기준 약 5~10분 소요됩니다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'microsoft/codebert-base'
BATCH_SIZE = 64  # GPU 메모리에 따라 조정

print(f'모델 로드 중... (device: {DEVICE})')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()
print('CodeBERT 로드 완료')

# 패키지명을 텍스트로 임베딩 (install_script가 없으면 name 사용)
if 'install_script' in df_balanced.columns:
    texts = df_balanced['install_script'].fillna('').tolist()
    print('install_script 컬럼 사용')
else:
    texts = df_balanced['name'].fillna('').tolist()
    print('패키지명 기반 임베딩')

def embed_batch(text_list, batch_size=BATCH_SIZE):
    all_emb = []
    total = len(text_list)
    t0 = time.time()
    for i in range(0, total, batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt', max_length=512,
            truncation=True, padding=True
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = bert_model(**inputs)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_emb.append(cls)
        done = min(i+batch_size, total)
        elapsed = time.time() - t0
        eta = elapsed / done * (total - done) if done > 0 else 0
        print(f'  {done}/{total}  경과 {elapsed:.0f}s  남은 예상 {eta:.0f}s', end='\r')
    print()
    return np.vstack(all_emb)

print(f'\nCodeBERT 임베딩 시작: {len(texts)}건')
emb_X = embed_batch(texts)
print(f'임베딩 완료: {emb_X.shape}')

## 5. 피처 결합 & XGBoost 학습

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error, classification_report

# 결합 (768 + 9 = 777차원)
X = np.hstack([emb_X, meta_X])
y = df_balanced['label'].values
print(f'최종 피처 행렬: {X.shape}')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=neg/pos,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',     # GPU 학습
    device='cuda' if torch.cuda.is_available() else 'cpu',
    eval_metric='logloss',
    random_state=RANDOM_SEED,
    verbosity=1,
)

model.fit(
    X_train_s, y_train,
    eval_set=[(X_val_s, y_val)],
    verbose=50,
)
print('학습 완료')

## 6. 성능 평가

In [ ]:
y_pred  = model.predict(X_val_s)
y_proba = model.predict_proba(X_val_s)[:, 1]

f1  = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_proba)
mae = mean_absolute_error(y_val, y_proba)
fp  = ((y_pred == 1) & (y_val == 0)).sum()
fpr = fp / (y_val == 0).sum()

print('=' * 50)
print(f'  F1-Score  : {f1:.4f}  (베이스라인 0.9565)')
print(f'  AUC-ROC   : {auc:.4f}  (베이스라인 0.9917)')
print(f'  MAE       : {mae:.4f}')
print(f'  FPR (오탐): {fpr:.4f}')
print('=' * 50)
print(classification_report(y_val, y_pred, target_names=['benign','malicious']))

## 7. 모델 저장 & 다운로드

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

bundle = {
    'model':      model,
    'scaler':     scaler,
    'metrics':    {'f1': f1, 'auc': auc, 'mae': mae},
    'model_type': 'ensemble_codebert_xgboost',
    'feat_dim':   X.shape[1],
}

PKL_PATH = 'models/ensemble_xgb.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(bundle, f)

print(f'모델 저장 완료: {PKL_PATH}')
print(f'파일 크기: {os.path.getsize(PKL_PATH) / 1024 / 1024:.1f} MB')

In [ ]:
# Colab에서 로컬로 다운로드
from google.colab import files
files.download(PKL_PATH)
print('다운로드 완료 — models/ 폴더에 넣어주세요')